# 10 — Final test evaluation: sentiment & priority

**The one time test is opened.** Configurations were selected on dev in
[`03_benchmark_sentiment.ipynb`](03_benchmark_sentiment.ipynb),
[`04_benchmark_priority.ipynb`](04_benchmark_priority.ipynb) and
[`06_improve_sentiment.ipynb`](06_improve_sentiment.ipynb). Nothing is chosen here by
looking at test.

**Protocol**

1. **Ground truth is the dataset's own labels** — the `sentiment` / `priority` columns in
   `datasets/<lang>/{train,test}_labeled.csv` (v5 prompt labels). The 500-row gold benchmark
   set is *not* an evaluation target anywhere in this notebook; §1 proves it is disjoint from
   test.
2. Final models are refit on **train+dev** (all 9,998 ids in the official train file, fanned
   to five languages) and scored once on the official BANKING77 test file.
3. Dev is scored too, from a train-only fit, so the dev → test gap is visible rather than
   assumed.
4. Confidence intervals resample over **ticket `id`** — the five language copies of one
   ticket are one observation, not five.

**No intent classifier appears here.** The `intent-chained` / `intent-lookup` priority
baselines from the bake-off are deliberately dropped.

In [1]:
import sys, json, time
from pathlib import Path

REPO = Path.cwd()
while not (REPO / "ml" / "swiftbench").exists() and REPO != REPO.parent:
    REPO = REPO.parent
sys.path.insert(0, str(REPO / "ml"))

import numpy as np
import pandas as pd

import swiftbench as sb
from swiftbench import config, data, imbalance, metrics, models, results, splits

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda v: f"{v:.4f}")

AUTHOR = "sithija"
LANGS = config.LANGUAGES

manifest = splits.ensure()
print("split sha:", manifest["sha"], manifest["counts"])

split sha: e7b5934392cd {'train': 8500, 'dev': 1498, 'test': 3079}


## 1. Integrity — is test clean?

Three things have to hold before any number below means anything.

**The gold 500 must not be in test.** `id` is only unique *within* a file: the train file
numbers 0–10002 and the test file numbers 0–3079 are independent sequences, so ~3,076 ids
collide numerically while referring to different tickets. Membership is therefore checked on
**text**, not on id.

**Labels must be id-aligned across the five languages**, since sentiment/priority are labeled
once in English and copied across.

**Train and test must not share text.**

In [2]:
tr_file = data.load_language("english", "train")
te_file = data.load_language("english", "test")
gold = pd.read_csv(REPO / "datasets" / "english" / "500_benchmarkset.csv")

norm = lambda s: set(s.astype(str).str.strip())
gold_text, tr_text, te_text = norm(gold["text"]), norm(tr_file["text_en"]), norm(te_file["text_en"])

align = data.check_alignment()
mismatch = int(align[["category_mismatch", "sentiment_mismatch", "priority_mismatch"]].to_numpy().sum())

checks = {
    "gold rows":                       len(gold),
    "gold texts found in TRAIN file":  len(gold_text & tr_text),
    "gold texts found in TEST file":   len(gold_text & te_text),
    "train/test id collisions":        len(set(tr_file.id) & set(te_file.id)),
    "train/test TEXT overlap":         len(tr_text & te_text),
    "cross-language label mismatches": mismatch,
}
for k, v in checks.items():
    print(f"{k:34s} {v}")

assert checks["gold texts found in TEST file"] == 0,  "gold benchmark leaked into test"
assert checks["gold texts found in TRAIN file"] == len(gold), "gold rows are not all in train"
assert mismatch == 0, "labels are not id-aligned across languages"
print("\nPASS — the gold set sits entirely inside train, and test is untouched by it.")
print("      id collisions are a numbering artefact, not shared tickets (text overlap is ~0).")

gold rows                          500
gold texts found in TRAIN file     500
gold texts found in TEST file      0
train/test id collisions           3076
train/test TEXT overlap            6
cross-language label mismatches    0

PASS — the gold set sits entirely inside train, and test is untouched by it.
      id collisions are a numbering artefact, not shared tickets (text overlap is ~0).


In [3]:
# Class balance differs between dev and test -- this matters in section 3.
rows = []
for portion in ("dev", "test"):
    d = splits.get(["english"], portion)          # english alone == one row per ticket
    rows.append({
        "portion": portion, "tickets": len(d),
        "negative": int((d.sentiment == "Negative").sum()),
        "negative %": 100 * (d.sentiment == "Negative").mean(),
        **{f"{p} %": 100 * (d.priority == p).mean() for p in config.PRIORITY_LABELS},
    })
prevalence = pd.DataFrame(rows)
display(prevalence)
print("Test is ~1.3pp less Negative than dev. Section 3 measures what that costs on its own.")

,portion,tickets,negative,negative %,Low %,Medium %,High %
0,dev,1498,68,4.5394,52.7370,37.1162,10.1469
1,test,3079,101,3.2803,54.8230,35.6934,9.4836


Test is ~1.3pp less Negative than dev. Section 3 measures what that costs on its own.


## 2. The evaluation

`multi` regime throughout — one model trained on all five languages, per §1 of the bake-off
report. Two fits per configuration:

| fit on | scored on | what it tells us |
|---|---|---|
| train (8,500 ids) | dev | reproduces the selection-time number |
| train+dev (9,998 ids) | **test** | the number that ships |

`headline` is `negative_f1` for sentiment and `macro_f1` for priority — never accuracy.

In [4]:
# Selected on dev, frozen before test was opened.
SENTIMENT_CONFIGS = [("tfidf-svm", "class_weight", 0.5), ("tfidf-logreg", "ros", 3.0)]
PRIORITY_CONFIGS  = [("tfidf-svm", "class_weight", 1.0), ("tfidf-logreg", "class_weight", 1.0)]

TRAIN_ONLY = splits.get(LANGS, "train")
TRAIN_DEV  = data.load_languages(LANGS, "train")   # every id in the official train file
assert len(TRAIN_DEV) == 9998 * len(LANGS), len(TRAIN_DEV)
print(f"train-only {len(TRAIN_ONLY):,} rows      train+dev {len(TRAIN_DEV):,} rows")


def fit_model(task, model, arm, C, train_df):
    label_col = data.label_column(task)
    fit = imbalance.resample(train_df, label_col, arm)
    clf = models.build(model, class_weight=imbalance.class_weight_for(arm), C=C)
    clf.fit(fit[config.TEXT_COLUMN], fit[label_col])
    return clf

train-only 42,500 rows      train+dev 49,990 rows


In [5]:
# --- fast headline scorer + id-grouped bootstrap -------------------------------
# metrics.score is the source of truth; these are numeric shortcuts for the 1,000-draw
# bootstrap loop only, and are asserted equal to it before use.

def _codes(y, labels):
    lut = {l: i for i, l in enumerate(labels)}
    return np.array([lut.get(v, -1) for v in y], dtype=np.int8)

def _f1(tp, fp, fn):
    return 0.0 if tp == 0 else 2 * tp / (2 * tp + fp + fn)

def fast_headline(yt, yp, task):
    # negative_f1 for sentiment; macro_f1 over the 3 fixed labels for priority.
    if task == "sentiment":
        t, p = yt == 1, yp == 1                     # 1 == Negative
        return _f1((t & p).sum(), (~t & p).sum(), (t & ~p).sum())
    return float(np.mean([
        _f1(((yt == k) & (yp == k)).sum(), ((yt != k) & (yp == k)).sum(), ((yt == k) & (yp != k)).sum())
        for k in range(3)
    ]))

LABELS = {"sentiment": ["Neutral", "Negative"], "priority": config.PRIORITY_LABELS}

def boot_ci(y_true, y_pred, task, ids, n_boot=1000, seed=42):
    # Percentile CI resampled over unique ticket id; all 5 language copies travel together.
    labs = LABELS[task]
    yt, yp = _codes(y_true, labs), _codes(y_pred, labs)
    groups = [g.to_numpy() for _, g in pd.Series(np.arange(len(yt))).groupby(np.asarray(ids))]
    rng = np.random.default_rng(seed)
    n = len(groups)
    stats = np.empty(n_boot)
    for b in range(n_boot):
        idx = np.concatenate([groups[i] for i in rng.integers(0, n, n)])
        stats[b] = fast_headline(yt[idx], yp[idx], task)
    lo, hi = np.percentile(stats, [2.5, 97.5])
    return {"ci_low": float(lo), "ci_high": float(hi), "ci_width": float(hi - lo), "n_tickets": n}

# sanity: the shortcut must agree with metrics.score
_d = splits.get(LANGS, "dev")
_m = fit_model("sentiment", "tfidf-svm", "class_weight", 0.5, TRAIN_ONLY)
_p = _m.predict(_d[config.TEXT_COLUMN])
_a = metrics.score(_d.sentiment, _p, "sentiment")["negative_f1"]
_b = fast_headline(_codes(_d.sentiment, LABELS["sentiment"]), _codes(_p, LABELS["sentiment"]), "sentiment")
assert abs(_a - _b) < 1e-12, (_a, _b)
print(f"fast scorer agrees with metrics.score: {_a:.6f} == {_b:.6f}")

fast scorer agrees with metrics.score: 0.614362 == 0.614362


In [6]:
def evaluate(task, configs, save_test=True):
    label_col = data.label_column(task)
    rows, fitted = [], {}
    for model, arm, C in configs:
        for fit_portion, train_df, eval_portion in [("train", TRAIN_ONLY, "dev"),
                                                    ("train+dev", TRAIN_DEV, "test")]:
            t0 = time.time()
            clf = fit_model(task, model, arm, C, train_df)
            fitted[(task, model, arm, fit_portion)] = clf
            pooled = splits.get(LANGS, eval_portion)
            pred_all = clf.predict(pooled[config.TEXT_COLUMN])

            for lang in LANGS + ["ALL"]:
                m = np.ones(len(pooled), bool) if lang == "ALL" else (pooled.language == lang).to_numpy()
                yt, yp = pooled[label_col].to_numpy()[m], pred_all[m]
                sc = metrics.score(yt, yp, task)
                row = {"task": task, "model": model, "arm": arm, "C": C,
                       "fit": fit_portion, "eval": eval_portion, "lang": lang,
                       "headline_metric": sc["headline_metric"],
                       **{k: v for k, v in sc.items() if not isinstance(v, str)}}
                if lang == "ALL":
                    row.update(boot_ci(yt, yp, task, pooled["id"].to_numpy()[m]))
                rows.append(row)

            if eval_portion == "test" and save_test:
                sc = metrics.score(pooled[label_col], pred_all, task)
                results.save(task, model, LANGS, "all", arm, "test", sc, author=AUTHOR,
                             extra={"regime": "multi", "fit_portion": "train+dev", "C": C})
            h = rows[-1]["headline"]
            print(f"  {model:13s} {arm:12s} fit={fit_portion:9s} -> {eval_portion:4s} "
                  f"ALL {sc['headline_metric']}={h:.4f}   [{time.time()-t0:.0f}s]")
    return pd.DataFrame(rows), fitted

print("=== sentiment ===");  sent, sent_fits = evaluate("sentiment", SENTIMENT_CONFIGS)
print("=== priority ===");   prio, prio_fits = evaluate("priority",  PRIORITY_CONFIGS)
res = pd.concat([sent, prio], ignore_index=True)

=== sentiment ===


  tfidf-svm     class_weight fit=train     -> dev  ALL negative_f1=0.6144   [3s]


  tfidf-svm     class_weight fit=train+dev -> test ALL negative_f1=0.4572   [4s]


  tfidf-logreg  ros          fit=train     -> dev  ALL negative_f1=0.5933   [5s]


  tfidf-logreg  ros          fit=train+dev -> test ALL negative_f1=0.4225   [7s]
=== priority ===


  tfidf-svm     class_weight fit=train     -> dev  ALL macro_f1=0.9028   [4s]


  tfidf-svm     class_weight fit=train+dev -> test ALL macro_f1=0.8722   [5s]


  tfidf-logreg  class_weight fit=train     -> dev  ALL macro_f1=0.8972   [5s]


  tfidf-logreg  class_weight fit=train+dev -> test ALL macro_f1=0.8683   [6s]


In [7]:
show = ["model", "arm", "fit", "eval", "lang", "headline", "accuracy", "ci_low", "ci_high", "n_tickets"]
print("SENTIMENT — pooled (negative_f1)")
display(sent[sent.lang == "ALL"][show + ["negative_precision", "negative_recall", "n_negative_true"]])
print("PRIORITY — pooled (macro_f1)")
display(prio[prio.lang == "ALL"][show + ["f1_low", "f1_medium", "f1_high"]])

SENTIMENT — pooled (negative_f1)


,model,arm,fit,eval,lang,headline,accuracy,ci_low,ci_high,n_tickets,negative_precision,negative_recall,n_negative_true
5,tfidf-svm,class_weight,train,dev,ALL,0.6144,0.9613,0.5383,0.6852,1498.0000,0.5607,0.6794,340
11,tfidf-svm,class_weight,train+dev,test,ALL,0.4572,0.9522,0.3970,0.5130,3079.0000,0.3643,0.6139,505
17,tfidf-logreg,ros,train,dev,ALL,0.5933,0.9581,0.5134,0.6676,1498.0000,0.5301,0.6735,340
23,tfidf-logreg,ros,train+dev,test,ALL,0.4225,0.9521,0.3627,0.4788,3079.0000,0.3493,0.5347,505


PRIORITY — pooled (macro_f1)


,model,arm,fit,eval,lang,headline,accuracy,ci_low,ci_high,n_tickets,f1_low,f1_medium,f1_high
5,tfidf-svm,class_weight,train,dev,ALL,0.9028,0.9093,0.8887,0.9165,1498.0000,0.9252,0.8918,0.8914
11,tfidf-svm,class_weight,train+dev,test,ALL,0.8722,0.8871,0.8605,0.8831,3079.0000,0.9117,0.8574,0.8476
17,tfidf-logreg,class_weight,train,dev,ALL,0.8972,0.9069,0.8824,0.9121,1498.0000,0.9235,0.8929,0.8751
23,tfidf-logreg,class_weight,train+dev,test,ALL,0.8683,0.8863,0.8557,0.8802,3079.0000,0.9126,0.8593,0.8330


In [8]:
# Per-language, test only -- where each language actually lands.
piv = (res[(res["eval"] == "test") & (res.lang != "ALL")]
       .pivot_table(index=["task", "model", "arm"], columns="lang", values="headline"))
print("Per-language TEST headline (sentiment=negative_f1, priority=macro_f1)")
display(piv)

Per-language TEST headline (sentiment=negative_f1, priority=macro_f1)


lang                                 english  singlish  sinhala  tamil  tamilish
task      model        arm                                                      
priority  tfidf-logreg class_weight   0.8908    0.8839   0.8822 0.8847    0.8002
          tfidf-svm    class_weight   0.9032    0.8915   0.8745 0.8905    0.7994
sentiment tfidf-logreg ros            0.4361    0.4170   0.3843 0.5018    0.3523
          tfidf-svm    class_weight   0.4582    0.4615   0.4138 0.5252    0.4229

## 3. Why test is lower than dev

Sentiment drops from ~0.63 (dev) to ~0.46 (test). Two candidate causes, and they are not the
same thing:

- **Selection optimism** — dev picked this configuration, so dev flatters it.
- **Prevalence shift** — test is 3.28% Negative against dev's 4.54%. At a fixed recall, rarer
  positives mean more false positives per true one, so precision and therefore F1 fall *even
  for an identical model*.

The second is measurable: hold the model and threshold fixed, and resample test down to dev's
Negative rate by dropping Neutral **tickets** (all five language copies together). Whatever gap
survives is the real generalization loss.

In [9]:
def prevalence_matched(task, clf, target_rate, n_draws=25, seed=42):
    # Score clf on test repeatedly, subsampling Neutral ids to hit target_rate.
    test = splits.get(LANGS, "test")
    label_col = data.label_column(task)
    per_ticket = test[test.language == "english"][["id", label_col]]
    neg_ids = per_ticket.loc[per_ticket[label_col] == "Negative", "id"].to_numpy()
    neu_ids = per_ticket.loc[per_ticket[label_col] != "Negative", "id"].to_numpy()

    keep_neu = int(round(len(neg_ids) / target_rate)) - len(neg_ids)
    pred = pd.Series(clf.predict(test[config.TEXT_COLUMN]), index=test.index)
    rng = np.random.default_rng(seed)

    out = []
    for _ in range(n_draws):
        ids = np.concatenate([neg_ids, rng.choice(neu_ids, keep_neu, replace=False)])
        m = test["id"].isin(ids).to_numpy()
        out.append(metrics.score(test[label_col].to_numpy()[m], pred.to_numpy()[m], task)["headline"])
    return float(np.mean(out)), float(np.std(out)), keep_neu + len(neg_ids)

dev_rate = float((splits.get(["english"], "dev").sentiment == "Negative").mean())
clf = sent_fits[("sentiment", "tfidf-svm", "class_weight", "train+dev")]

dev_score  = float(sent.query("model=='tfidf-svm' and eval=='dev'  and lang=='ALL'").headline.iloc[0])
test_score = float(sent.query("model=='tfidf-svm' and eval=='test' and lang=='ALL'").headline.iloc[0])
matched, sd, n_tick = prevalence_matched("sentiment", clf, dev_rate)

print(f"dev  Negative rate {dev_rate:.4%}")
print(f"dev  negative_f1                          {dev_score:.4f}")
print(f"test negative_f1 (3.28% Negative)         {test_score:.4f}")
print(f"test negative_f1 @ dev prevalence         {matched:.4f} +/- {sd:.4f}   ({n_tick} tickets/draw)")
print()
print(f"  total dev -> test drop      {dev_score - test_score:+.4f}")
print(f"  attributable to prevalence  {matched - test_score:+.4f}")
print(f"  genuine generalization loss {dev_score - matched:+.4f}")

dev  Negative rate 4.5394%
dev  negative_f1                          0.6144
test negative_f1 (3.28% Negative)         0.4572
test negative_f1 @ dev prevalence         0.5165 +/- 0.0065   (2225 tickets/draw)

  total dev -> test drop      +0.1571
  attributable to prevalence  +0.0592
  genuine generalization loss +0.0979


## 4. Decision threshold

Every number above uses the estimator's default cut. `06_improve_sentiment.ipynb` found tuning
it worth ~+0.027 — larger than most gaps the bake-off ranked models on.

The threshold is tuned by **5-fold CV over train+dev, folds drawn on `id`**, never on test.
Scores are the raw `decision_function` / `predict_proba` value, not the min-max normalised
version in `tuning._positive_scores`: min-max is fit- and dataset-specific, so a threshold
tuned on a fold would mean something different on test. Raw scores are absolute and transfer.

In [10]:
from sklearn.model_selection import StratifiedKFold

def raw_scores(pipe, X, pos="Negative"):
    est, feats = pipe[-1], pipe[:-1]
    Z = feats.transform(X)
    if hasattr(est, "predict_proba"):
        return est.predict_proba(Z)[:, list(est.classes_).index(pos)], 0.5
    s = est.decision_function(Z)
    if s.ndim > 1:
        s = s[:, list(est.classes_).index(pos)]
    elif list(est.classes_)[1] != pos:
        s = -s
    return s, 0.0

pool_ids = sorted(set(manifest["train_ids"]) | set(manifest["dev_ids"]))
eng = data.load_language("english", "train").set_index("id")
strat = eng.loc[pool_ids, "sentiment"].values
frames = {l: data.load_language(l, "train") for l in LANGS}
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=config.RANDOM_STATE)

cv_rows = []
for model, arm, C in SENTIMENT_CONFIGS:
    for fold, (a, b) in enumerate(kf.split(pool_ids, strat)):
        tr_ids, va_ids = {pool_ids[i] for i in a}, {pool_ids[i] for i in b}
        tr = pd.concat([f[f.id.isin(tr_ids)] for f in frames.values()], ignore_index=True)
        va = pd.concat([f[f.id.isin(va_ids)] for f in frames.values()], ignore_index=True)
        clf = fit_model("sentiment", model, arm, C, tr)
        s, default_t = raw_scores(clf, va[config.TEXT_COLUMN])
        yt = _codes(va.sentiment, LABELS["sentiment"])
        grid = np.quantile(s, np.linspace(0.80, 0.999, 200))
        f1s = np.array([fast_headline(yt, (s >= t).astype(np.int8), "sentiment") for t in grid])
        best = int(f1s.argmax())
        cv_rows.append({"model": model, "arm": arm, "fold": fold,
                        "tuned_t": float(grid[best]), "tuned_f1": float(f1s[best]),
                        "default_f1": fast_headline(yt, (s >= default_t).astype(np.int8), "sentiment")})
        print(f"  {model:13s} fold{fold}  t={grid[best]:+.4f}  f1={f1s[best]:.4f} "
              f"(default {cv_rows[-1]['default_f1']:.4f})")

cv = pd.DataFrame(cv_rows)
display(cv.groupby(["model", "arm"])[["tuned_t", "tuned_f1", "default_f1"]].agg(["mean", "std"]))

  tfidf-svm     fold0  t=-0.0411  f1=0.5645 (default 0.5605)


  tfidf-svm     fold1  t=+0.2063  f1=0.5922 (default 0.5837)


  tfidf-svm     fold2  t=+0.1056  f1=0.5313 (default 0.5255)


  tfidf-svm     fold3  t=+0.0832  f1=0.5762 (default 0.5647)


  tfidf-svm     fold4  t=+0.0421  f1=0.5712 (default 0.5698)


...

  tfidf-logreg  fold1  t=+0.5756  f1=0.5762 (default 0.5684)


  tfidf-logreg  fold2  t=+0.4688  f1=0.5176 (default 0.5108)


  tfidf-logreg  fold3  t=+0.5926  f1=0.5731 (default 0.5624)


  tfidf-logreg  fold4  t=+0.4911  f1=0.5468 (default 0.5465)


tuned_t        tuned_f1        default_f1       
                             mean    std     mean    std       mean    std
model        arm                                                          
tfidf-logreg ros           0.5119 0.0695   0.5544 0.0237     0.5485 0.0226
tfidf-svm    class_weight  0.0792 0.0905   0.5671 0.0224     0.5608 0.0216

In [11]:
# Apply the CV-chosen threshold to test. Test never informed the choice.
thr = cv.groupby(["model", "arm"]).tuned_t.mean().to_dict()
test_pool = splits.get(LANGS, "test")
yt_test = test_pool.sentiment.to_numpy()

tuned_rows = []
for model, arm, C in SENTIMENT_CONFIGS:
    clf = sent_fits[("sentiment", model, arm, "train+dev")]
    s, default_t = raw_scores(clf, test_pool[config.TEXT_COLUMN])
    for name, t in [("default", default_t), ("cv-tuned", thr[(model, arm)])]:
        yp = np.where(s >= t, "Negative", "Neutral")
        sc = metrics.score(yt_test, yp, "sentiment")
        tuned_rows.append({"model": model, "arm": arm, "threshold": name, "t": round(float(t), 4),
                           **{k: v for k, v in sc.items() if not isinstance(v, str)},
                           **boot_ci(yt_test, yp, "sentiment", test_pool["id"].to_numpy())})

tuned = pd.DataFrame(tuned_rows)
display(tuned[["model", "arm", "threshold", "t", "negative_f1", "negative_precision",
               "negative_recall", "accuracy", "ci_low", "ci_high"]])

,model,arm,threshold,t,negative_f1,negative_precision,negative_recall,accuracy,ci_low,ci_high
0,tfidf-svm,class_weight,default,0.0000,0.4572,0.3643,0.6139,0.9522,0.3970,0.5130
1,tfidf-svm,class_weight,cv-tuned,0.0792,0.4524,0.3775,0.5644,0.9552,0.3885,0.5094
2,tfidf-logreg,ros,default,0.5000,0.4225,0.3493,0.5347,0.9521,0.3627,0.4788
3,tfidf-logreg,ros,cv-tuned,0.5119,0.4287,0.3587,0.5327,0.9534,0.3684,0.4849


## 5. Read against the label ceiling

`05_label_ceiling.ipynb` measured how well the v5 prompt labels — what every model here is
trained and scored against — agree with human annotation on the gold 500.

| task | v5 vs human | κ |
|---|---|---|
| sentiment | 0.5769 negative-F1, CI [0.40, 0.73] | 0.55 |
| priority | 0.7722 macro-F1 | 0.64 |

This does **not** cap the scores above — against v5 labels a model can in principle reach 1.0.
It caps their *meaning*. A priority model at 0.87 has learned the v5 labeling rule well; the
rule itself only matches human judgement at 0.77, so 0.77 is the operational ceiling regardless
of what the model scores. For sentiment the model sits inside the ceiling's own CI, which is
why the recommendation is to improve labels (v6 already scores 0.6875 vs v5's 0.4615 on the
250-row holdout) rather than only to chase the model.

In [12]:
OUT = REPO / "ml" / "reports"
res.to_csv(OUT / "final_test_results.csv", index=False)
tuned.to_csv(OUT / "final_test_thresholds.csv", index=False)
cv.to_csv(OUT / "final_test_threshold_cv.csv", index=False)
prevalence.to_csv(OUT / "final_test_prevalence.csv", index=False)

best = (res[(res["eval"] == "test") & (res.lang == "ALL")]
        .sort_values(["task", "headline"], ascending=[True, False])
        .groupby("task").head(1))
print("SHIPPING CANDIDATES (test, pooled, default threshold)")
display(best[["task", "model", "arm", "headline", "headline_metric", "accuracy", "ci_low", "ci_high"]])
print("wrote final_test_results.csv, final_test_thresholds.csv, "
      "final_test_threshold_cv.csv, final_test_prevalence.csv")

SHIPPING CANDIDATES (test, pooled, default threshold)


,task,model,arm,headline,headline_metric,accuracy,ci_low,ci_high
35,priority,tfidf-svm,class_weight,0.8722,macro_f1,0.8871,0.8605,0.8831
11,sentiment,tfidf-svm,class_weight,0.4572,negative_f1,0.9522,0.3970,0.5130


wrote final_test_results.csv, final_test_thresholds.csv, final_test_threshold_cv.csv, final_test_prevalence.csv
